# AIM-Flow Kaggle Demo

This notebook clones the GitHub repo, installs dependencies, authenticates with Hugging Face, loads SD3 Medium, and compares base SD3 vs AIM-Flow.

In [ ]:
# User variables
GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/aim-flow.git"
HF_TOKEN = ""  # user fills this or uses Kaggle secret
PROMPT_KEY = "cyborg_dogs"
OUTPUT_DIR = "/kaggle/working/aim_flow_outputs"

In [ ]:
# Clone repo
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
# Install dependencies
!pip install -q -r requirements-kaggle.txt

In [ ]:
# Hugging Face login/token handling
import os

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
else:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found. Make sure you have accepted SD3 Medium license and added HF_TOKEN as a Kaggle secret.")

In [ ]:
# Check GPU
import torch

print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

In [ ]:
# Run the comparison
!python scripts/run_compare.py \
  --config configs/sd3_medium_kaggle.yaml \
  --prompts configs/sample_prompts.yaml \
  --prompt-key {PROMPT_KEY} \
  --output-dir {OUTPUT_DIR} \
  --modes base anchor naive full

In [ ]:
# Display grid
from PIL import Image
import matplotlib.pyplot as plt

grid_path = f"{OUTPUT_DIR}/comparison_grid.png"
img = Image.open(grid_path)
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis("off")

In [ ]:
# Show metadata
import glob
import json

for path in glob.glob(f"{OUTPUT_DIR}/*.json"):
    print("\n", path)
    with open(path) as f:
        data = json.load(f)
    print(json.dumps(data, indent=2)[:2000])

## Notes

- If out of memory, reduce steps to 16, use 384x384, or enable CPU offload.
- If model access fails, accept the model license on Hugging Face and provide HF_TOKEN.
- This prototype avoids VQA/reward models and only compares images qualitatively.